# RoomBeacon — Auditable Bronze-to-Silver Processing Report

This executable report explains the canonical helper without duplicating it. Runtime evidence shows rules, lineage, before/after values, unresolved conditions, quality gates, and publication results.

**Canonical Silver dataset:** `data/silver/rental_listings.parquet`  
**Metadata:** `data/silver/rental_listings.metadata.json`  
**DuckDB role:** analytical/query engine; no physical Silver table is required for persistence.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sys
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'crawler').is_dir() and (p/'analytics').is_dir())
sys.path.insert(0, str(PROJECT_ROOT))
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT/'.env', override=False)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from analytics.duckdb.connection import create_analytics_connection, resolve_runtime_path
from roombeacon_crawler.config.get_env import env
from notebooks.utils.notebook_audit import load_snapshot
from notebooks.utils.silver_processing import build_silver_dataset, evaluate_pre_silver_quality_gate
from notebooks.utils.silver_reporting import (
    status_summary, text_change_summary, changed_text_examples, sample_status_rows,
    numeric_audit_report, iqr_threshold_diagnostics, flag_reason_breakdown,
    multi_flag_distribution, duplicate_group_summary,
)
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 90)
plt.style.use('seaborn-v0_8-whitegrid')
COLOR='#4C78A8'; REVIEW='#E45756'; OK='#59A14F'; MUTED='#9D9D9D'

def barh_table(frame, label, value, title, xlabel='Rows', color=COLOR):
    plot=frame.sort_values(value)
    fig, ax=plt.subplots(figsize=(9, max(3.2, .42*len(plot))))
    bars=ax.barh(plot[label].astype(str), plot[value], color=color)
    ax.set(title=title, xlabel=xlabel, ylabel='')
    ax.bar_label(bars, fmt='{:,.0f}', padding=3, fontsize=8)
    plt.tight_layout(); plt.show()

def grouped_coverage(frame, title):
    x=np.arange(len(frame)); width=.36
    fig, ax=plt.subplots(figsize=(9,4.5))
    ax.bar(x-width/2, frame['Raw Available'], width, label='Raw Available', color=MUTED)
    ax.bar(x+width/2, frame['Clean Available'], width, label='Clean Available', color=COLOR)
    ax.set(title=title, xlabel='Field', ylabel='Rows', xticks=x, xticklabels=frame['Field'])
    ax.legend(); plt.tight_layout(); plt.show()

## Silver Processing Pipeline Overview

```mermaid
flowchart TD
 B[MySQL Bronze via DuckDB] --> T[build_silver_dataset]
 T --> S[silver_df] --> G{Pre-Silver Quality Gate}
 G -->|PASS| P[DuckDB COPY to Parquet]
 P --> F[(data/silver/rental_listings.parquet)]
 F --> D[03_roombeacon_processing.ipynb]
```

The diagram is explanatory. `build_silver_dataset()` remains the canonical transformation. DuckDB remains available for Bronze access, diagnostics, and analytical queries; the Parquet file is the persistent clean checkpoint.

## 01. Load Bronze Snapshot

> **Purpose:** Freeze one Bronze-backed runtime snapshot and aligned numeric evidence.

### Actual Rule
One non-null, unique `rental_post_id` per input row. Raw fields remain immutable.


In [ ]:
conn=create_analytics_connection()
bronze_df, raw_evidence, RUN_CONTEXT=load_snapshot(conn, PROJECT_ROOT)
BRONZE_COLUMNS=bronze_df.columns.tolist(); RAW_FINGERPRINT=bronze_df.copy(deep=True)
assert len(bronze_df)>0 and bronze_df.rental_post_id.notna().all() and bronze_df.rental_post_id.is_unique
display(pd.Series(RUN_CONTEXT, name='Runtime Value').to_frame())


## 02. Processing Contract Matrix

> **Purpose:** Make every transformation boundary and raw-field preservation promise explicit.


In [ ]:
processing_contract=pd.DataFrame([
 ['Text Standardization','title_raw; full_address_text; location_raw; best_address_text','*_clean','—','No'],
 ['Title Quality','title_clean','title_quality_status','title_quality_status','No'],
 ['Address Parsing','best_address_text_clean','street/ward/district/province_text_extracted','parse_status','No'],
 ['Ward Mapping','ward_text_extracted; district_text_extracted','ward_normalized; ward_current; candidates','ward_mapping_status','No'],
 ['Price Validation','price_amount + aligned price_raw evidence','price_amount_clean; lineage/regression','price_quality_status','No'],
 ['Area Validation','area_value + aligned area_raw evidence','area_value_clean; lineage/regression','area_quality_status','No'],
 ['Coordinate Quality','map_latitude; map_longitude; map_provider; full_address_text','trust evidence; has_trusted_coordinate','coordinate_quality_status','No'],
 ['Duplicate Audit','title/address clean + price/area clean','duplicate_candidate_group','duplicate_candidate_status','No'],
 ['Temporal Quality','first/last/latest_observed_at','—','temporal_quality_status','No'],
],columns=['Processing Step','Raw / Input Fields','Output Fields','Status Field','Raw Mutated?'])
display(processing_contract)


## 03. General Text Standardization

> **Purpose:** Produce safe, comparable text representations while retaining every raw string.

### Actual Rule
The shared helper applies Unicode NFC, removes invisible/control and configured decorative characters, normalizes whitespace and safe punctuation, and converts empty results to `NULL`.

### Input → Output
`title_raw`, `full_address_text`, `location_raw`, `best_address_text` → corresponding `*_clean` fields.


In [ ]:
silver_df=build_silver_dataset(bronze_df, raw_evidence)
TEXT_FIELDS={'Title':('title_raw','title_clean'),'Full Address':('full_address_text','full_address_text_clean'),'Location':('location_raw','location_raw_clean'),'Best Address':('best_address_text','best_address_text_clean')}
text_summary=text_change_summary(bronze_df,silver_df,TEXT_FIELDS)
display(text_summary)
barh_table(text_summary,'Field','Changed Rows','Text Values Changed by Deterministic Standardization')
grouped_coverage(text_summary,'Raw vs Clean Text Coverage')
display(Markdown('### Before → After Examples'))
display(changed_text_examples(bronze_df,silver_df,TEXT_FIELDS,20))
display(Markdown('**Remaining Issues:** cleanup standardizes representation only; it does not infer missing text.'))


## 04. Title Quality Processing

> **Purpose:** Identify titles that are usable for downstream inspection.

### Actual Rule
`MISSING` when `title_clean` is null; `TOO_SHORT` when non-null length is below 5 characters; otherwise `USABLE`.

### Input → Output
`title_raw` → `title_clean` → `title_quality_status`


In [ ]:
title_definitions=pd.DataFrame([
 ['USABLE','title_clean is present and length ≥ 5','Text is structurally usable','Eligible subject to other fields'],
 ['TOO_SHORT','title_clean length < 5','Insufficient title evidence','Requires review'],
 ['MISSING','title_clean is NULL','No usable title','Requires review'],
],columns=['Status','Actual Rule','Meaning','Downstream Implication'])
title_status=status_summary(silver_df.title_quality_status,'Status')
display(title_definitions); display(title_status)
barh_table(title_status,'Status','Rows','Title Quality Status')
display(sample_status_rows(silver_df,'title_quality_status',['TOO_SHORT','MISSING'],['rental_post_id','source_code','title_raw','title_clean','title_quality_status'],5))


## 05. Address Standardization & Parsing

> **Purpose:** Standardize the best available address and expose parser evidence without guessing.

### Actual Rule and Input → Output
```text
best_address_text → best_address_text_clean → Address Parser
  ├─ street_text_extracted
  ├─ ward_text_extracted
  ├─ district_text_extracted
  ├─ province_text_extracted
  └─ parse_status
```
Parser statuses reflect actual extracted components and recognition evidence.


In [ ]:
address_summary=text_change_summary(bronze_df,silver_df,{'Best Address':('best_address_text','best_address_text_clean')})
display(address_summary); display(changed_text_examples(bronze_df,silver_df,{'Best Address':('best_address_text','best_address_text_clean')},12))
components={'Street':'street_text_extracted','Ward':'ward_text_extracted','District':'district_text_extracted','Province':'province_text_extracted'}
address_coverage=pd.DataFrame([{'Parsed Field':label,'Available Rows':int(silver_df[col].notna().sum()),'Missing Rows':int(silver_df[col].isna().sum()),'Coverage (%)':round(silver_df[col].notna().mean()*100,2)} for label,col in components.items()])
parse_status=status_summary(silver_df.parse_status,'Status')
display(address_coverage); display(parse_status)
barh_table(parse_status,'Status','Rows','Address Parse Status')
address_examples=sample_status_rows(silver_df,'parse_status',['PARSED','PARTIALLY_PARSED','UNRECOGNIZED_FORMAT','NO_ADDRESS_EVIDENCE'],['best_address_text','best_address_text_clean','street_text_extracted','ward_text_extracted','district_text_extracted','province_text_extracted','parse_status'],4)
display(Markdown('### Before → After Examples')); display(address_examples)
display(Markdown('**Remaining Issues:** parser statuses describe observed evidence; they do not certify an official address.'))


## 06. Administrative Unit Mapping

> **Purpose:** Resolve extracted ward text through the existing internal RoomBeacon reference.

### Actual Rule and Input → Output
```text
ward_text_extracted → ward_normalized → internal reference → ward_current → ward_mapping_status
```
Ambiguous candidates remain unresolved; the helper never forces `ward_current`.


In [ ]:
ward_status=status_summary(silver_df.ward_mapping_status,'Mapping Status'); display(ward_status)
barh_table(ward_status,'Mapping Status','Rows','Ward Mapping Status')
ward_examples=sample_status_rows(silver_df,'ward_mapping_status',['MAPPED','AMBIGUOUS','UNMAPPED'],['best_address_text','ward_text_extracted','district_text_extracted','ward_normalized','ward_current','ward_mapping_status'],5)
display(Markdown('### Before → After Examples')); display(ward_examples)
display(Markdown('**Remaining Issues:** `ward_current` is an internal reference result, not independently verified administrative truth.'))


## 07. Price Validation and Parser Comparison Hardening

> **Purpose:** Accept only lineage-aligned existing values or narrowly safe deterministic recovery, while keeping parser disagreement explicit in canonical Silver.

### Actual Rule and Input → Output
```text
Raw Price Evidence → Existing Numeric Price → Reparsed Price → comparison
→ Final Clean Price → price_quality_status + price_parser_comparison_status
```
`MATCH`, `DISAGREEMENT`, `REPARSED_ONLY`, `EXISTING_ONLY`, `NO_PRICE_EVIDENCE`, and `UNCOMPARABLE` describe only available comparison evidence. A disagreement does not select a winner: the existing canonical acceptance rule still controls `price_amount_clean`.


In [ ]:
price_audit=numeric_audit_report(bronze_df,raw_evidence,silver_df,'price')
price_defs=pd.DataFrame([
 ['VALIDATED_EXISTING','Existing value accepted; lineage and VND/month unit contract pass','Clean value keeps existing numeric value'],
 ['REPARSE_ACCEPTED_CLEAN','Existing value is null; aligned raw syntax reparses and passes domain/unit gates','Clean value receives reparsed value'],
 ['MISSING_OR_REVIEW','No accepted existing or recovery path','Clean value remains null'],
 ['UNKNOWN_LINEAGE','Raw evidence is not aligned to the contributing version','Clean value remains null'],
],columns=['Status','Actual Rule','Clean Value Behavior'])
comparison_defs=pd.DataFrame([
 ['MATCH','Existing and reparsed values are both available and equal'],['DISAGREEMENT','Both values are available and differ; clean value is not silently replaced'],['REPARSED_ONLY','Only reparsed numeric evidence is available'],['EXISTING_ONLY','Only existing numeric evidence is available'],['NO_PRICE_EVIDENCE','Neither numeric value nor non-empty raw evidence is available'],['UNCOMPARABLE','Raw evidence exists but neither comparable numeric value is available']],columns=['Price Parser Comparison Status','Actual Rule'])
display(price_defs); display(comparison_defs)
price_status=status_summary(silver_df.price_quality_status,'Status'); comparison_status=status_summary(silver_df.price_parser_comparison_status,'Price Parser Comparison Status')
display(price_status); display(comparison_status)
barh_table(price_status,'Status','Rows','Price Validation Status')
barh_table(comparison_status,'Price Parser Comparison Status','Rows','Price Parser Comparison Status vs Rows')
price_metrics=pd.DataFrame({'Metric':['Total Rows','Raw Price Available','Existing Numeric Available','Reparse Available','Reparse Accepted','Parser Agreement','Parser Disagreement','Missing / No Raw','Negotiable','Parse Failed / Reparse NULL','Requires Review / Unavailable'], 'Rows':[len(price_audit),price_audit.raw.notna().sum(),price_audit.existing.notna().sum(),price_audit.reparsed.notna().sum(),price_audit.action.eq('REPARSE_ACCEPTED_CLEAN').sum(),silver_df.price_parser_comparison_status.eq('MATCH').sum(),silver_df.price_parser_comparison_status.eq('DISAGREEMENT').sum(),silver_df.price_parser_comparison_status.eq('NO_PRICE_EVIDENCE').sum(),price_audit.semantic.eq('NEGOTIABLE_NON_NUMERIC').sum(),price_audit.regression.eq('REPARSE_NULL').sum(),silver_df.price_quality_status.isin(['MISSING_OR_REVIEW','UNKNOWN_LINEAGE']).sum()]})
price_metrics['Percentage (%)']=(price_metrics.Rows/len(price_audit)*100).round(2); display(price_metrics)
price_cov=pd.DataFrame({'Stage':['Existing Numeric Price','Final Clean Price'],'Rows':[price_audit.existing.notna().sum(),silver_df.price_amount_clean.notna().sum()]}); barh_table(price_cov,'Stage','Rows','Price Coverage Before vs After')
price_examples=price_audit.assign(source_code=bronze_df.source_code,rental_post_id=bronze_df.rental_post_id,price_amount_clean=silver_df.price_amount_clean,price_validation_status=silver_df.price_quality_status,price_parser_comparison_status=silver_df.price_parser_comparison_status)
disagreement_examples=price_examples.loc[price_examples.price_parser_comparison_status.eq('DISAGREEMENT'),['source_code','rental_post_id','raw','existing','reparsed','price_amount_clean','price_validation_status','price_parser_comparison_status']].head(15).rename(columns={'raw':'raw price evidence','existing':'existing numeric price','reparsed':'reparsed price'})
display(Markdown('### Parser Disagreement Examples and Final Clean-value Behavior')); display(disagreement_examples)
comparable=price_examples.loc[price_examples.existing.notna()&price_examples.reparsed.notna()].copy(); focus=comparable.loc[pd.to_numeric(comparable.existing).le(pd.to_numeric(comparable.existing).quantile(.99)) & pd.to_numeric(comparable.reparsed).le(pd.to_numeric(comparable.reparsed).quantile(.99))].head(8000)
fig,ax=plt.subplots(figsize=(7,5)); ax.scatter(pd.to_numeric(focus.existing),pd.to_numeric(focus.reparsed),s=10,alpha=.3,c=np.where(focus.price_parser_comparison_status.eq('DISAGREEMENT'),REVIEW,COLOR)); ax.set(title='Existing vs Reparsed Price — Visualization-only Central 99% Range',xlabel='Existing Numeric Price (VND/month)',ylabel='Reparsed Price (VND/month)'); plt.tight_layout(); plt.show()


## 08. Area Validation

> **Purpose:** Apply the same lineage-safe validation pattern to area evidence.

### Actual Rule and Input → Output
`Raw Area Evidence → Existing Area → Reparsed Area → Final Clean Area → Area Status`. Existing positive values are retained only with aligned lineage; recovery requires safe full-string area syntax and a positive parsed value.


In [ ]:
area_audit=numeric_audit_report(bronze_df,raw_evidence,silver_df,'area')
area_defs=pd.DataFrame([
 ['VALIDATED_EXISTING','Positive existing area with aligned lineage','Keep existing value'],['REPARSE_ACCEPTED_CLEAN','Null existing area; aligned safe syntax reparses positive','Use reparsed value'],['MISSING_OR_REVIEW','No accepted existing or recovery path','Remain null'],['UNKNOWN_LINEAGE','Evidence alignment cannot be established','Remain null']],columns=['Status','Actual Rule','Clean Value Behavior'])
display(area_defs)
area_status=status_summary(silver_df.area_quality_status,'Status'); display(area_status)
area_metrics=pd.DataFrame({'Metric':['Total Rows','Raw Area Available','Existing Area Available','Reparsed Area','Accepted Reparse','Parser Agreement','Parser Disagreement','Missing / Review'],'Rows':[len(area_audit),area_audit.raw.notna().sum(),area_audit.existing.notna().sum(),area_audit.reparsed.notna().sum(),area_audit.action.eq('REPARSE_ACCEPTED_CLEAN').sum(),area_audit.regression.eq('CANONICAL_MATCH').sum(),area_audit.regression.eq('DIFFERENT_REVIEW').sum(),silver_df.area_quality_status.isin(['MISSING_OR_REVIEW','UNKNOWN_LINEAGE']).sum()]}); area_metrics['Percentage (%)']=(area_metrics.Rows/len(area_audit)*100).round(2); display(area_metrics)
barh_table(area_status,'Status','Rows','Area Validation Status')
area_cov=pd.DataFrame({'Stage':['Existing Area','Final Clean Area'],'Rows':[area_audit.existing.notna().sum(),silver_df.area_value_clean.notna().sum()]}); barh_table(area_cov,'Stage','Rows','Area Coverage Before vs After')
area_outcomes=pd.DataFrame({'Outcome':['Recovered','Parser Disagreement','Parse Failure','Unavailable / Review'],'Rows':[area_audit.action.eq('REPARSE_ACCEPTED_CLEAN').sum(),area_audit.regression.eq('DIFFERENT_REVIEW').sum(),area_audit.regression.eq('REPARSE_NULL').sum(),silver_df.area_quality_status.isin(['MISSING_OR_REVIEW','UNKNOWN_LINEAGE']).sum()]}); barh_table(area_outcomes,'Outcome','Rows','Area Recovery and Review Outcomes')
area_examples=area_audit.assign(source_code=bronze_df.source_code,rental_post_id=bronze_df.rental_post_id)
area_examples['Example Category']=np.select([area_examples.action.eq('VALIDATED_EXISTING'),area_examples.action.eq('REPARSE_ACCEPTED_CLEAN'),area_examples.regression.eq('DIFFERENT_REVIEW'),area_examples.regression.eq('REPARSE_NULL')],['Validated Existing','Accepted Reparse','Parser Disagreement','Parse Failure'],default='Missing / Review')
display(area_examples.groupby('Example Category',group_keys=False).head(3)[['Example Category','source_code','rental_post_id','raw','existing','reparsed','final_clean','quality_status']])


## 09. Numeric Outlier Flags

> **Purpose:** Flag statistical tails without deleting or changing any value.

### Actual Rule
For each clean numeric field, activate only with at least four values and positive finite IQR. Flag values below `Q1 − 1.5×IQR` or above `Q3 + 1.5×IQR`.


In [ ]:
outlier_thresholds=pd.concat([iqr_threshold_diagnostics(silver_df.price_amount_clean,'Price'),iqr_threshold_diagnostics(silver_df.area_value_clean,'Area')],ignore_index=True); display(outlier_thresholds)
outlier_status=status_summary(silver_df.numeric_outlier_status,'Status'); display(outlier_status); barh_table(outlier_status,'Status','Rows','Numeric Outlier Flag Status')
price_plot=pd.to_numeric(silver_df.price_amount_clean,errors='coerce').dropna(); price_limit=price_plot.quantile(.99)
fig,ax=plt.subplots(figsize=(9,3.8)); ax.boxplot(price_plot[price_plot.le(price_limit)],vert=False); ax.set(title='Price Distribution — Visualization-only Central 99% Range',xlabel='Clean Price (VND/month)',yticks=[]); plt.tight_layout(); plt.show()
area_plot=pd.to_numeric(silver_df.area_value_clean,errors='coerce').dropna(); area_limit=area_plot.quantile(.99)
fig,ax=plt.subplots(figsize=(9,4)); ax.hist(area_plot[area_plot.le(area_limit)],bins=40,color=COLOR); ax.set(title='Area Distribution — Visualization-only Central 99% Range',xlabel='Clean Area (m²)',ylabel='Rows'); plt.tight_layout(); plt.show()
outlier_examples=silver_df.loc[silver_df.numeric_outlier_status.ne('NOT_OUTLIER'),['rental_post_id','source_code','title_raw','price_amount','price_amount_clean','area_value','area_value_clean','best_address_text','numeric_outlier_status']].head(20); display(outlier_examples)
display(Markdown('**Remaining Issues:** an IQR flag is statistical review evidence, not proof that a listing is wrong. No row is dropped.'))


## 10. Coordinate Trust Classification

> **Purpose:** Distinguish availability, geometric validity, provider trust, and repeated-point conflicts.

### Actual Rule and Input → Output
```text
latitude / longitude → pair available? → finite/in range/non-zero?
→ trusted provider? → repeated coordinate? → address consistency?
→ coordinate_trust_reason → coordinate_quality_status
```
Canonical `INVALID` includes missing, non-finite, out-of-bounds, and zero pairs. The presentation uses the more specific existing `coordinate_trust_reason`; no Silver semantics are changed.


In [ ]:
coordinate_reason=status_summary(silver_df.coordinate_trust_reason,'Coordinate Status'); display(coordinate_reason); barh_table(coordinate_reason,'Coordinate Status','Rows','Coordinate Trust Reason')
coord_metrics=pd.DataFrame({'Metric':['Complete Coordinate Pairs','Missing Coordinate Pairs','Invalid Range / Non-finite / Zero','Unique Pairs','Repeated Pair Records','Largest Repeated Group','Usable','Untrusted'], 'Rows':[(silver_df.map_latitude.notna()&silver_df.map_longitude.notna()).sum(),silver_df.coordinate_trust_reason.eq('MISSING_COORDINATE_PAIR').sum(),silver_df.coordinate_trust_reason.isin(['NONFINITE_COORDINATE','OUT_OF_BOUNDS','ZERO_COORDINATE']).sum(),silver_df.coordinate_pair_listing_count.eq(1).sum(),silver_df.coordinate_pair_listing_count.gt(1).sum(),silver_df.coordinate_pair_listing_count.max(),silver_df.has_trusted_coordinate.sum(),(~silver_df.has_trusted_coordinate & silver_df.coordinate_pair_valid).sum()]}); display(coord_metrics)
repeated=silver_df.loc[silver_df.coordinate_pair_listing_count.gt(1),'coordinate_pair_listing_count']
fig,ax=plt.subplots(figsize=(8,4)); ax.hist(repeated,bins=min(30,max(5,repeated.nunique())),color=COLOR); ax.set(title='Repeated Coordinate Group-size Distribution',xlabel='Listings at Rounded Coordinate',ylabel='Rows'); plt.tight_layout(); plt.show()
coord_groups=(silver_df.loc[silver_df.coordinate_pair_listing_count.gt(1)].assign(Latitude=lambda x:x.map_latitude.round(6),Longitude=lambda x:x.map_longitude.round(6)).groupby(['Latitude','Longitude'],dropna=True).agg(Listings=('rental_post_id','size'),Distinct_Addresses=('full_address_text','nunique'),Distinct_Wards=('ward_current','nunique'),Distinct_Sources=('source_code','nunique'),Coordinate_Status=('coordinate_trust_reason',lambda s:', '.join(sorted(set(s.dropna().astype(str)))))).reset_index().sort_values('Listings',ascending=False))
top_coord=coord_groups.head(15).copy(); top_coord['Coordinate']=top_coord.Latitude.astype(str)+', '+top_coord.Longitude.astype(str)
barh_table(top_coord,'Coordinate','Listings','Top Repeated Coordinates')
display(top_coord[['Latitude','Longitude','Listings','Distinct_Addresses','Distinct_Wards','Distinct_Sources','Coordinate_Status']])


## 11. Cross-field Consistency — Price × Area Hardening

> **Purpose:** Separate data availability from relationship-quality review.

### Actual Rule
`price_area_availability_status` distinguishes `AVAILABLE`, `MISSING_PRICE`, `MISSING_AREA`, and `MISSING_BOTH` using canonical clean fields. For available pairs, `price_area_quality_status` applies existing price/area IQR flags first, then a 1.5×IQR audit of `price_amount_clean / area_value_clean`. `CHECKED_NO_FLAG` means these deterministic checks found no flag—not that the relationship is universally “consistent.” The ratio is an audit metric only and is not persisted.


In [ ]:
availability_status=status_summary(silver_df.price_area_availability_status,'Price × Area Availability Status')
quality_status=status_summary(silver_df.price_area_quality_status,'Price × Area Quality Status')
display(availability_status); display(quality_status)
barh_table(quality_status,'Price × Area Quality Status','Rows','Price × Area Quality Status')
price_per_area_audit=pd.to_numeric(silver_df.price_amount_clean,errors='coerce')/pd.to_numeric(silver_df.area_value_clean,errors='coerce').where(pd.to_numeric(silver_df.area_value_clean,errors='coerce').gt(0))
ratio_thresholds=iqr_threshold_diagnostics(price_per_area_audit,'Price per Area Audit'); display(ratio_thresholds)
review_examples=silver_df.assign(price_per_area_audit=price_per_area_audit).loc[silver_df.price_area_quality_status.ne('CHECKED_NO_FLAG'),['title_clean','price_amount_clean','area_value_clean','price_per_area_audit','numeric_outlier_status','price_area_availability_status','price_area_quality_status','best_address_text_clean','source_code']].groupby('price_area_quality_status',group_keys=False).head(4)
display(Markdown('### Representative Non-normal Price × Area States')); display(review_examples)
pa=silver_df.assign(price_per_area_audit=price_per_area_audit).loc[silver_df.price_area_availability_status.eq('AVAILABLE')].head(8000)
fig,ax=plt.subplots(figsize=(7,5)); flagged=pa.price_area_quality_status.ne('CHECKED_NO_FLAG'); ax.scatter(pa.area_value_clean,pa.price_amount_clean,s=10,alpha=.28,c=np.where(flagged,REVIEW,COLOR)); ax.set(title='Price × Area Audit (Flagged States Highlighted)',xlabel='Clean Area (m²)',ylabel='Clean Price (VND/month)'); plt.tight_layout(); plt.show()


## 12. Duplicate / Repost Candidate Hardening

> **Purpose:** Extend conservative candidate coverage without fuzzy matching, merging, or row deletion.

### Actual Blocking and Matching Rules
1. `EXACT_FINGERPRINT`: same non-null normalized title, normalized address, clean price, and clean area.
2. `SAME_ADDRESS_PRICE_AREA`: cross-source only; same non-null normalized address, clean price, and clean area.
3. `SAME_TITLE_PRICE_WARD`: cross-source only; same non-null normalized title, clean price, and current ward.

Candidate blocks are linked into deterministic connected components; no all-pairs comparison occurs. `duplicate_scope` is `SAME_SOURCE`, `CROSS_SOURCE`, or `NOT_APPLICABLE`. `POSSIBLE_DUPLICATE` remains candidate evidence, never confirmation. Missing values and shared coordinates alone are not evidence.


In [ ]:
duplicate_status=status_summary(silver_df.duplicate_candidate_status,'Status'); display(duplicate_status); barh_table(duplicate_status,'Status','Rows','Duplicate Candidate Status')
duplicate_groups,duplicate_distribution=duplicate_group_summary(silver_df)
duplicate_metrics=pd.Series({'Candidate Groups':len(duplicate_groups),'Candidate Listings':int(silver_df.duplicate_candidate_status.eq('POSSIBLE_DUPLICATE').sum()),'Largest Candidate Group':int(duplicate_groups.Listings.max()) if len(duplicate_groups) else 0,'Same-source Groups':int(duplicate_groups['Candidate Scope'].eq('Same-source').sum()) if len(duplicate_groups) else 0,'Cross-source Groups':int(duplicate_groups['Candidate Scope'].eq('Cross-source').sum()) if len(duplicate_groups) else 0},name='Value').to_frame(); display(duplicate_metrics)
if len(duplicate_distribution):
 fig,ax=plt.subplots(figsize=(8,4)); ax.bar(duplicate_distribution['Group Size'].astype(str),duplicate_distribution['Candidate Groups'],color=COLOR); ax.set(title='Duplicate Candidate Group-size Distribution',xlabel='Listings per Candidate Group',ylabel='Candidate Groups'); plt.tight_layout(); plt.show()
 scope=duplicate_groups['Candidate Scope'].value_counts().rename_axis('Scope').reset_index(name='Groups'); barh_table(scope,'Scope','Groups','Same-source vs Cross-source Candidate Groups')
for scope_name in ['SAME_SOURCE','CROSS_SOURCE']:
 ids=silver_df.loc[silver_df.duplicate_scope.eq(scope_name),'duplicate_candidate_group'].dropna().drop_duplicates().head(3)
 sample=silver_df.loc[silver_df.duplicate_candidate_group.isin(ids),['duplicate_candidate_group','duplicate_scope','duplicate_match_reason','source_code','source_listing_id','title_clean','price_amount_clean','area_value_clean','best_address_text_clean','ward_current']]
 display(Markdown(f'### {scope_name.replace("_"," ").title()} Candidate Examples')); display(sample)
if not silver_df.duplicate_scope.eq('CROSS_SOURCE').any(): display(Markdown('**No cross-source candidates were found under the current conservative rules.**'))


## 13. Temporal Semantics Validation

> **Purpose:** Flag timestamp ordering that cannot be safely explained by current evidence.

### Actual Rule
Missing first/last/latest timestamp → `INSUFFICIENT_DATA`. `first > last`, `latest < first`, or `latest > last` → `REQUIRES_REVIEW`; otherwise `VALID`. Cause is not inferred.


In [ ]:
temporal_status=status_summary(silver_df.temporal_quality_status,'Status'); display(temporal_status); barh_table(temporal_status,'Status','Rows','Temporal Quality Status')
display(silver_df.loc[silver_df.temporal_quality_status.eq('REQUIRES_REVIEW'),['rental_post_id','source_code','first_observed_at','last_observed_at','latest_observed_at','active_days','temporal_quality_status']].head(15))
display(Markdown('**Remaining Issues:** Requires semantic review; crawler or backfill history is not sufficient here to prove corruption.'))


## 14. Final Row Quality Status

> **Purpose:** Explain the roll-up without converting flags into a quality score.

### Actual Rule
`REQUIRES_REVIEW` remains limited to non-usable title, temporal review, or ambiguous ward mapping. Otherwise `READY_WITH_FLAGS` includes existing flags plus explicit price parser disagreement and Price×Area review states. A disagreement does not automatically become critical review because the existing accepted clean-price contract remains unchanged.


In [ ]:
row_status=status_summary(silver_df.row_quality_status,'Row Status')
row_defs=pd.DataFrame([['READY','No critical review condition and no configured flags'],['READY_WITH_FLAGS','No critical condition, but at least one configured quality flag/unavailable field'],['REQUIRES_REVIEW','Non-usable title, temporal review, or ambiguous ward mapping']],columns=['Row Status','Definition']).merge(row_status,on='Row Status',how='left'); display(row_defs); barh_table(row_status,'Row Status','Rows','Final Row Quality Status')
display(Markdown('### Top Flag Reasons'))
flag_breakdown,flag_matrix=flag_reason_breakdown(silver_df); display(flag_breakdown)
barh_table(flag_breakdown.head(15),'Flag Reason','Affected Rows','Top Flag Reasons')
display(Markdown('### Number of Flags per Row'))
flag_distribution=multi_flag_distribution(flag_matrix); display(flag_distribution)
fig,ax=plt.subplots(figsize=(7,4)); bars=ax.bar(flag_distribution['Flag Count'],flag_distribution.Rows,color=COLOR); ax.set(title='Number of Diagnostic Flags per Row',xlabel='Flag Count (not a score)',ylabel='Rows'); ax.bar_label(bars,fmt='{:,.0f}',padding=3); plt.tight_layout(); plt.show()


## 15. Silver Dataset Contract and Raw → Silver Lineage Examples

> **Purpose:** Make every output field and representative transformation trace readable.


In [ ]:
def field_group(field):
 if field in ['source_code','rental_post_id','source_listing_id','url']: return 'Identity'
 if 'title' in field: return 'Title'
 if 'price' in field: return 'Price'
 if 'area' in field: return 'Area'
 if field in ['full_address_text','location_raw','best_address_text','best_address_source','full_address_inherited'] or 'address' in field or field in ['street_text_extracted','parse_status']: return 'Address'
 if 'ward' in field or 'district' in field or 'province' in field: return 'Administrative Location'
 if field.startswith('map_') or 'coordinate' in field or field=='has_trusted_coordinate': return 'Coordinate'
 if 'duplicate' in field: return 'Duplicate'
 if 'observed' in field or field=='active_days' or 'temporal' in field: return 'Temporal'
 if 'quality' in field or 'outlier' in field or 'cross_field' in field: return 'Quality'
 if 'lineage' in field or 'regression' in field: return 'Lineage'
 return 'Other'
def field_kind(field):
 if field in BRONZE_COLUMNS: return 'Raw'
 if field.endswith('_status') or field.endswith('_flag') or field in ['parse_status','has_trusted_coordinate']: return 'Status'
 return 'Clean / Derived'
contract_rows=[]
for field in silver_df.columns:
 contract_rows.append({'Group':field_group(field),'Field':field,'Origin':'Bronze snapshot' if field in BRONZE_COLUMNS else 'Canonical Silver helper','Meaning':field.replace('_',' ').title(),'Raw / Clean / Status':field_kind(field),'Nullable?':'Yes' if silver_df[field].isna().any() else 'No','Notes':'Raw preserved unchanged' if field in BRONZE_COLUMNS else 'See rule section above'})
silver_contract=pd.DataFrame(contract_rows).sort_values(['Group','Field']); display(silver_contract)
selectors=[('Normal Clean',silver_df.row_quality_status.eq('READY')),('Price Reparse',silver_df.price_quality_status.eq('REPARSE_ACCEPTED_CLEAN')),('Ward Mapped',silver_df.ward_mapping_status.eq('MAPPED')),('Coordinate Review',~silver_df.has_trusted_coordinate),('Duplicate Candidate',silver_df.duplicate_candidate_status.eq('POSSIBLE_DUPLICATE')),('Temporal Review',silver_df.temporal_quality_status.eq('REQUIRES_REVIEW'))]
chosen=[]
for label,mask in selectors:
 for idx in silver_df.index[mask]:
  if idx not in chosen: chosen.append(idx); break
lineage=silver_df.loc[chosen,['rental_post_id','source_code','title_raw','title_clean','price_amount','price_amount_clean','area_value','area_value_clean','best_address_text','best_address_text_clean','ward_text_extracted','ward_current','coordinate_trust_reason','duplicate_candidate_status','temporal_quality_status','row_quality_status']].copy()
lineage.insert(0,'Lineage Example',[next((label for label,mask in selectors if bool(mask.loc[idx])), 'Representative') for idx in chosen])
display(Markdown('### Raw → Silver Lineage Examples')); display(lineage)


## 16. Pre-Silver Quality Gate

> **Purpose:** Stop publication when grain, identity, lineage, clean-value, or documented-status invariants fail.

The asserts remain authoritative and materialization has not occurred before this section.


In [ ]:
quality_gate=evaluate_pre_silver_quality_gate(RAW_FINGERPRINT,silver_df)
protections={'row_count_preserved':'Protects against silent row deletion','rental_post_id_non_null':'Protects identity completeness','rental_post_id_unique':'Protects one-row grain','source_identity_preserved':'Protects source identity','raw_source_fields_unchanged':'Protects Bronze/source lineage','no_row_multiplication_or_deletion':'Protects join cardinality','price_clean_contract_valid':'Protects price acceptance rules','area_clean_contract_valid':'Protects area acceptance rules','ambiguous_ward_not_truth':'Prevents guessed administrative mapping','invalid_coordinates_not_usable':'Protects spatial trust','outliers_preserved_and_flagged':'Prevents silent outlier deletion','duplicate_candidates_preserved_and_flagged':'Prevents silent candidate deletion','price_parser_disagreement_preserved':'Keeps parser disagreements explicit','price_area_status_domain_valid':'Protects Price × Area status domains','duplicate_scope_domain_valid':'Protects duplicate scope vocabulary','duplicate_candidate_rows_preserved':'Prevents candidate-driven row deletion','cross_source_duplicate_rule_deterministic':'Protects conservative cross-source rule semantics','documented_status_values_only':'Protects status contract'}
quality_gate_frame=quality_gate.to_frame().rename(columns={'invariant':'Invariant','passed':'Passed'})
quality_gate_frame['PASS / FAIL']=np.where(quality_gate_frame.Passed,'PASS','FAIL'); quality_gate_frame['What It Protects']=quality_gate_frame.Invariant.map(protections)
display(quality_gate_frame[['Invariant','PASS / FAIL','What It Protects']])
gate_counts=quality_gate_frame['PASS / FAIL'].value_counts().rename_axis('Result').reset_index(name='Invariants')
fig,ax=plt.subplots(figsize=(6,3.5)); bars=ax.bar(gate_counts.Result,gate_counts.Invariants,color=[OK if x=='PASS' else REVIEW for x in gate_counts.Result]); ax.set(title='Pre-Silver Quality Gate Results',xlabel='Result',ylabel='Invariants'); ax.bar_label(bars,padding=3); plt.tight_layout(); plt.show()
assert quality_gate.passed


## 17. Save Canonical Silver Dataset

> **Purpose:** Persist the validated `silver_df` to `data/silver/rental_listings.parquet` only after the Pre-Silver Quality Gate passes.

```text
silver_df
    ↓
Quality Gate PASS
    ↓
DuckDB COPY to Parquet
    ↓
data/silver/rental_listings.parquet
```

The temporary file is read back and validated before it atomically replaces the canonical file.

In [ ]:
assert quality_gate.passed

SILVER_DIR = resolve_runtime_path(env.processing.silver_dir)
SILVER_PATH = SILVER_DIR / 'rental_listings.parquet'
METADATA_PATH = SILVER_DIR / 'rental_listings.metadata.json'
TEMP_SILVER_PATH = SILVER_DIR / 'rental_listings.parquet.tmp'
TEMP_METADATA_PATH = SILVER_DIR / 'rental_listings.metadata.json.tmp'
SILVER_DIR.mkdir(parents=True, exist_ok=True)

for temporary_path in (TEMP_SILVER_PATH, TEMP_METADATA_PATH):
    temporary_path.unlink(missing_ok=True)

try:
    temp_silver_sql_path = str(TEMP_SILVER_PATH).replace("'", "''")
    conn.register('_canonical_silver_write', silver_df)
    try:
        conn.execute(f"COPY (SELECT * FROM _canonical_silver_write) TO '{temp_silver_sql_path}' (FORMAT PARQUET)")
    finally:
        conn.unregister('_canonical_silver_write')
    saved_silver_df = conn.execute(
        f"SELECT * FROM read_parquet('{temp_silver_sql_path}')"
    ).df()
    assert len(saved_silver_df) == len(silver_df)
    assert saved_silver_df.rental_post_id.notna().all()
    assert saved_silver_df.rental_post_id.nunique() == len(saved_silver_df)
    assert list(saved_silver_df.columns) == list(silver_df.columns)
    assert {'rental_post_id', 'source_code', 'price_parser_comparison_status'} <= set(saved_silver_df.columns)

    observed = pd.to_datetime(saved_silver_df.latest_observed_at, errors='coerce').dropna()
    silver_metadata = {
        'dataset_name': 'rental_listings',
        'layer': 'silver',
        'generated_at': datetime.now(timezone.utc).astimezone().isoformat(),
        'row_count': len(saved_silver_df),
        'distinct_rental_post_id': int(saved_silver_df.rental_post_id.nunique()),
        'column_count': len(saved_silver_df.columns),
        'columns': saved_silver_df.columns.tolist(),
        'min_observed_at': str(observed.min()) if not observed.empty else None,
        'max_observed_at': str(observed.max()) if not observed.empty else None,
        'schema_version': '2.0.0',
        'processing_version': '2.0.0',
        'source_snapshot': RUN_CONTEXT,
    }
    TEMP_METADATA_PATH.write_text(json.dumps(silver_metadata, indent=2, ensure_ascii=False), encoding='utf-8')
    TEMP_SILVER_PATH.replace(SILVER_PATH)
    TEMP_METADATA_PATH.replace(METADATA_PATH)
finally:
    TEMP_SILVER_PATH.unlink(missing_ok=True)
    TEMP_METADATA_PATH.unlink(missing_ok=True)

# Remove obsolete names only after canonical publication succeeds.
for obsolete_dir in {SILVER_DIR, PROJECT_ROOT / 'data' / 'silver'}:
    for obsolete_name in ('rental_latest.parquet', 'rental_latest.metadata.json'):
        (obsolete_dir / obsolete_name).unlink(missing_ok=True)

save_summary = pd.Series({
    'Canonical Silver Path': str(SILVER_PATH),
    'Rows': len(saved_silver_df),
    'Unique rental_post_id': saved_silver_df.rental_post_id.nunique(),
    'Columns': len(saved_silver_df.columns),
    'File Size (bytes)': SILVER_PATH.stat().st_size,
    'Metadata Path': str(METADATA_PATH),
    'Save Status': 'PASS',
}, name='Value').to_frame()
display(save_summary)

## 18. Final Silver Validation and Health Summary

> **Purpose:** Independently read the canonical Parquet checkpoint, verify row/identity/schema parity, and answer: “After Processing, what problems still remain in Silver?”

In [ ]:
silver_sql_path = str(SILVER_PATH).replace("'", "''")
canonical_silver_df = conn.execute(
    f"SELECT * FROM read_parquet('{silver_sql_path}')"
).df()
assert len(canonical_silver_df) == len(silver_df)
assert canonical_silver_df.rental_post_id.notna().all()
assert canonical_silver_df.rental_post_id.nunique() == len(canonical_silver_df)
assert list(canonical_silver_df.columns) == list(silver_df.columns)
assert {'rental_post_id', 'source_code', 'price_parser_comparison_status'} <= set(canonical_silver_df.columns)
assert METADATA_PATH.exists()

canonical_summary = pd.DataFrame([{
    'row_count': len(canonical_silver_df),
    'distinct_rental_post_id': canonical_silver_df.rental_post_id.nunique(),
    'column_count': len(canonical_silver_df.columns),
    'validation_status': 'PASS',
}])
display(canonical_summary)

health=pd.DataFrame([
 ['Title',silver_df.title_quality_status.eq('USABLE').sum(),silver_df.title_quality_status.eq('TOO_SHORT').sum(),silver_df.title_quality_status.eq('MISSING').sum(),'Structural title usability'],
 ['Price',silver_df.price_amount_clean.notna().sum(),silver_df.price_outlier_flag.sum(),silver_df.price_amount_clean.isna().sum(),'Validated or safe recovery only'],
 ['Area',silver_df.area_value_clean.notna().sum(),silver_df.area_outlier_flag.sum(),silver_df.area_value_clean.isna().sum(),'Validated or safe recovery only'],
 ['Address',silver_df.parse_status.eq('PARSED').sum(),silver_df.parse_status.isin(['PARTIALLY_PARSED','UNRECOGNIZED_FORMAT']).sum(),silver_df.parse_status.eq('NO_ADDRESS_EVIDENCE').sum(),'Parser evidence, not official truth'],
 ['Ward Mapping',silver_df.ward_current.notna().sum(),silver_df.ward_mapping_status.isin(['UNMAPPED','AMBIGUOUS']).sum(),silver_df.ward_mapping_status.eq('MISSING').sum(),'Internal RoomBeacon reference'],
 ['Coordinates',silver_df.has_trusted_coordinate.sum(),silver_df.coordinate_trust_reason.ne('MISSING_COORDINATE_PAIR').sum()-silver_df.has_trusted_coordinate.sum(),silver_df.coordinate_trust_reason.eq('MISSING_COORDINATE_PAIR').sum(),'Missing is separate from invalid/untrusted'],
 ['Price × Area',silver_df.price_area_quality_status.eq('CHECKED_NO_FLAG').sum(),silver_df.price_area_quality_status.str.endswith('REVIEW').sum(),silver_df.price_area_quality_status.eq('INSUFFICIENT_DATA').sum(),'Availability and deterministic audit flags are separate'],
 ['Duplicate',silver_df.duplicate_candidate_status.eq('UNIQUE_FINGERPRINT').sum(),silver_df.duplicate_candidate_status.eq('POSSIBLE_DUPLICATE').sum(),silver_df.duplicate_candidate_status.eq('INSUFFICIENT_DATA').sum(),'Candidates are not confirmations'],
 ['Temporal',silver_df.temporal_quality_status.eq('VALID').sum(),silver_df.temporal_quality_status.eq('REQUIRES_REVIEW').sum(),silver_df.temporal_quality_status.eq('INSUFFICIENT_DATA').sum(),'Review cause is not inferred'],
],columns=['Quality Dimension','Healthy / Available','Flagged / Review','Missing / Unavailable','Notes']); display(Markdown('### Final Silver Health Summary')); display(health)
review_burden=pd.DataFrame({'Remaining Issue':['Price Review','Area Review','Address Parse Failure','Ward Unmapped / Ambiguous','Coordinate Missing / Untrusted','Duplicate Candidate','Temporal Review','Cross-field Review'],'Rows':[silver_df.price_quality_status.isin(['MISSING_OR_REVIEW','UNKNOWN_LINEAGE']).sum(),silver_df.area_quality_status.isin(['MISSING_OR_REVIEW','UNKNOWN_LINEAGE']).sum(),silver_df.parse_status.isin(['UNRECOGNIZED_FORMAT','NO_ADDRESS_EVIDENCE']).sum(),silver_df.ward_mapping_status.isin(['UNMAPPED','AMBIGUOUS']).sum(),(~silver_df.has_trusted_coordinate).sum(),silver_df.duplicate_candidate_status.eq('POSSIBLE_DUPLICATE').sum(),silver_df.temporal_quality_status.eq('REQUIRES_REVIEW').sum(),silver_df.price_area_quality_status.str.endswith('REVIEW').sum()]})
barh_table(review_burden,'Remaining Issue','Rows','Remaining Silver Quality Burdens')
pd.testing.assert_frame_equal(silver_df[BRONZE_COLUMNS],RAW_FINGERPRINT)
conn.close()